In [1]:
import json
import os
import time
import pytz
import requests
from pathlib import Path
from datetime import datetime,timezone,timedelta
from scipy.optimize import curve_fit
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from dotenv import load_dotenv
from bot_template import BaseBot, OrderBook, Order, OrderRequest, OrderResponse, Trade, Side, Product

load_dotenv()

EXCHANGE_URL  = "http://ec2-52-49-69-152.eu-west-1.compute.amazonaws.com/"   
USERNAME = os.getenv("EXCHANGE_USERNAME")
PASSWORD = os.getenv("EXCHANGE_PASSWORD")

LONDON_TZ = pytz.timezone('Europe/London')

In [2]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

def get_weather(past_steps=96, forecast_steps=96):
    """伦敦15分钟天气预报, 返回一个包含以下列的DataFrame: 时间, 温度, 湿度."""
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "humidity": m["relative_humidity_2m"],
    })

In [3]:
def get_wx_spot():
    weather_df = get_weather()
    weather_df = weather_df[(weather_df["time"]<=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)))&(weather_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)))]
    weather_df["temperature"] = weather_df["temperature"]*9/5+32
    wx_spot = weather_df[weather_df["time"]==LONDON_TZ.localize(datetime(2026,3,1,12,0,0))]["temperature"] * weather_df[weather_df["time"]==LONDON_TZ.localize(datetime(2026,3,1,12,0,0))]["humidity"]
    return round(wx_spot.item())

def get_wx_sum():
    weather_df = get_weather()
    weather_df = weather_df[(weather_df["time"]<=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)))&(weather_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)))]
    weather_df["temperature"] = weather_df["temperature"]*9/5+32
    weather_df["temp_hum_product"] = weather_df["temperature"]*weather_df["humidity"]
    wx_sum = weather_df["temp_hum_product"].sum()/100
    return round(wx_sum.item())

In [4]:
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

def get_thames(limit=400):
    """获取威斯敏斯特地区泰晤士河的最新潮汐数据. 返回一个包含以下列的DataFrame: 时间, 水平"""
    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": limit},
    )
    resp.raise_for_status()
    items = resp.json().get("items", [])
    df = pd.DataFrame(items)[["dateTime", "value"]].rename(columns={"dateTime": "time", "value": "level"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/London")
    return df.sort_values("time").reset_index(drop=True)

In [ ]:
# 潮汐调和分析 - 5个主要分潮
# M2(12.42h), S2(12.00h), N2(12.66h), K1(23.93h), O1(25.82h)
TIDAL_CONSTITUENTS = [
    ('M2', 2*np.pi/12.4206),
    ('S2', 2*np.pi/12.0000),
    ('N2', 2*np.pi/12.6583),
    ('K1', 2*np.pi/23.9345),
    ('O1', 2*np.pi/25.8194),
]

def fit_tidal_harmonic(thames_df):
    """
    调和分析: h(t) = Z0 + sum_i [A_i*cos(w_i*t) + B_i*sin(w_i*t)]
    用最小二乘法拟合，返回 (Z0, [(A1,B1), (A2,B2), ...]) 和残差std.
    """
    t0 = thames_df['time'].min()
    hours = (thames_df['time'] - t0).dt.total_seconds().values / 3600
    y = thames_df['level'].values
    # 构建设计矩阵: [1, cos(w1*t), sin(w1*t), cos(w2*t), sin(w2*t), ...]
    cols = [np.ones(len(hours))]
    for _, w in TIDAL_CONSTITUENTS:
        cols.append(np.cos(w * hours))
        cols.append(np.sin(w * hours))
    X = np.column_stack(cols)
    # 最小二乘
    coeffs, residuals, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_pred = X @ coeffs
    res_std = float(np.std(y - y_pred))
    return coeffs, t0, res_std

def predict_tidal_level(coeffs, t0, target_time):
    """用调和系数预测指定时刻的水位"""
    h = (target_time - t0).total_seconds() / 3600
    val = coeffs[0]
    for i, (_, w) in enumerate(TIDAL_CONSTITUENTS):
        val += coeffs[1 + 2*i] * np.cos(w * h)
        val += coeffs[2 + 2*i] * np.sin(w * h)
    return float(val)

def predict_tidal_series(coeffs, t0, times):
    """批量预测一组时刻的水位"""
    hours = np.array([(t - t0).total_seconds() / 3600 for t in times])
    vals = np.full(len(hours), coeffs[0])
    for i, (_, w) in enumerate(TIDAL_CONSTITUENTS):
        vals += coeffs[1 + 2*i] * np.cos(w * hours)
        vals += coeffs[2 + 2*i] * np.sin(w * hours)
    return vals

# 保留旧函数兼容性
def sin_function(x, a, b, c, d):
    return a + b * np.sin(c * x + d)

def fit_thames_data(thames_df):
    start_time = thames_df['time'].min()
    hours = (thames_df['time'] - start_time).dt.total_seconds() / 3600
    x_data, y_data = hours.values, thames_df['level'].values
    p0 = [np.mean(y_data), np.std(y_data), 2*np.pi/12.4206, 0]
    popt, _ = curve_fit(sin_function, x_data, y_data, p0=p0, maxfev=5000)
    return tuple(popt)


In [6]:
def get_tide_spot():
    thames_df = get_thames()
    a,b,c,d = fit_thames_data(thames_df)
    predicted_level = sin_function((LONDON_TZ.localize(datetime(2026,3,1,12,0,0))-thames_df["time"].min()).total_seconds()/3600,a,b,c,d)
    tide_spot = abs(predicted_level)*1000
    return round(tide_spot.item())

def get_tide_swing():
    thames_df = get_thames()
    thames_df = thames_df[thames_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0))]
    a,b,c,d = fit_thames_data(thames_df)
    
    date_range = pd.date_range(start=max(thames_df['time'])+timedelta(minutes=15),end=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)),freq="15min",tz="Europe/London")
    thames_missing_df = pd.DataFrame({'time':date_range})
    hours_since_start = (thames_missing_df["time"]-thames_df["time"].min()).dt.total_seconds()/3600
    thames_missing_df["level"] = a+b*np.sin(c*hours_since_start.values+d)
    thames_df = pd.concat([thames_df,thames_missing_df], ignore_index=True).sort_values("time").drop_duplicates(subset=["time"], keep="last").reset_index(drop=True)

    thames_df["level_cm"] = round(thames_df["level"]*100)
    thames_df["diff_cm"] = thames_df["level_cm"].diff().abs().dropna().reset_index(drop=True)
    thames_df["put_payoff"] = np.maximum(0,20-thames_df["diff_cm"])
    thames_df["call_payoff"] = np.maximum(0,thames_df["diff_cm"]-25)
    thames_df["total_payoff"] = thames_df["put_payoff"]+thames_df["call_payoff"]

    return thames_df["total_payoff"].sum()


In [7]:
AERODATABOX_KEY = os.getenv("AERODATABOX_KEY")
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
AIRPORT = "LHR"

def fetch_flights(airport=AIRPORT,offset_minutes=-360,duration_minutes=720,filters:dict|None=None):
    """按相对时间窗口(从现在开始的偏移量)获取航班信息. 该API抓取次数有限制所以说要保留历史文档"""
    params = f"?offsetMinutes={offset_minutes}&durationMinutes={duration_minutes}&direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

def fetch_flights_range(airport=AIRPORT,from_local="2026-02-28T12:00",to_local="2026-02-29T00:00",filters: dict|None = None):
    """根据明确的本地时间范围(最大跨度为12小时)获取航班信息. 该API抓取次数有限制所以说要保留历史文档"""
    params = "?direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}/{from_local}/{to_local}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

In [8]:
flights_data_1 = fetch_flights_range(from_local="2026-02-28T12:00",to_local="2026-03-01T00:00")

In [9]:
flights_data_2 = fetch_flights_range(from_local="2026-03-01T00:00",to_local="2026-03-01T12:00")

In [10]:
def get_lhr_count():
    global flights_data_1, flights_data_2    
    lhr_count = len(flights_data_1.get('arrivals',[]))+len(flights_data_1.get('departures',[]))+len(flights_data_2.get('arrivals',[]))+len(flights_data_2.get('departures',[]))
    return lhr_count

In [ ]:
def parse_flight_time(flight_record, flight_type):
    """解析航班时间. 优先用实际跑道时间, 其次修订时间, 最后计划时间."""
    time_fields = ['runwayTime', 'revisedTime', 'scheduledTime']
    movement = flight_record['movement']
    utc_str = None
    for field in time_fields:
        if field in movement and movement[field].get('utc'):
            utc_str = movement[field]['utc']
            break
    if utc_str is None:
        raise ValueError('No valid time field found')
    dt = pytz.utc.localize(datetime.strptime(utc_str, '%Y-%m-%d %H:%MZ'))
    return dt.astimezone(LONDON_TZ)

# 保留旧名兼容
flight_time = parse_flight_time

def group_flights(flights_data, start_time, end_time):
    """按时间窗口统计到达/出发航班数"""
    arrivals = 0
    for arr in flights_data.get('arrivals', []):
        try:
            if start_time <= parse_flight_time(arr, 'arrival') < end_time:
                arrivals += 1
        except Exception:
            pass
    departures = 0
    for dep in flights_data.get('departures', []):
        try:
            if start_time <= parse_flight_time(dep, 'departure') < end_time:
                departures += 1
        except Exception:
            pass
    return arrivals, departures


In [12]:
def get_settlement_price(symbol):
    price_map = {
        "TIDE_SPOT": get_tide_spot,
        "TIDE_SWING": get_tide_swing,
        "WX_SPOT": get_wx_spot,
        "WX_SUM": get_wx_sum,
        "LHR_COUNT": get_lhr_count,
        "LHR_INDEX": get_lhr_index,
        "LON_ETF": get_lon_etf,
        "LON_FLY": get_lon_fly,
    }
    fn = price_map.get(symbol)
    return fn() if fn else 10.0

def get_lon_etf():
    return get_tide_spot()+get_wx_spot()+get_lhr_count()

def get_lon_fly():
    etf_price = get_lon_etf() 
    part1 = 2*max(6200-etf_price,0)
    part2 = max(etf_price-6200,0)
    part3 = -2*max(etf_price-6600,0)
    part4 = 3*max(etf_price-7000,0)
    return part1 + part2 + part3 + part4

In [ ]:
class FairValueEngine:
    """
    Fair value 引擎 v4:
    - 潮汐: 5分潮调和分析
    - LON_FLY: 分量蒙特卡洛 (TIDE_SPOT折叠正态, LHR_COUNT泊松, WX_SPOT正态)
    - 贝叶斯更新: 动态市场权重
    - WX_SPOT std: forecast horizon 时间加权误差
    """
    CACHE_TTL     = 60
    SETTLE_TIME   = datetime(2026, 3, 1, 12, 0, 0)
    SESSION_START = datetime(2026, 2, 28, 12, 0, 0)
    MC_SAMPLES    = 20000
    TOTAL_HOURS   = 24.0   # 比赛总时长

    def __init__(self):
        self._weather_cache = None;  self._weather_ts  = 0.0
        self._thames_cache  = None;  self._thames_ts   = 0.0
        self._tidal_coeffs  = None
        self._flights_cache = None
        self._drift_history = {}
        self._market_weight = {}

    def _get_weather(self):
        if time.time() - self._weather_ts > self.CACHE_TTL:
            self._weather_cache = get_weather(past_steps=96, forecast_steps=96)
            self._weather_ts = time.time()
        return self._weather_cache

    def _get_thames(self):
        if time.time() - self._thames_ts > self.CACHE_TTL:
            df = get_thames(limit=672)
            self._thames_cache  = df
            self._tidal_coeffs  = fit_tidal_harmonic(df)
            self._thames_ts = time.time()
        return self._thames_cache, self._tidal_coeffs

    def init_flights(self):
        if self._flights_cache is None:
            print('Fetching flight data (2 API calls)...')
            d1 = fetch_flights_range(from_local='2026-02-28T12:00', to_local='2026-03-01T00:00')
            d2 = fetch_flights_range(from_local='2026-03-01T00:00', to_local='2026-03-01T12:00')
            self._flights_cache = (d1, d2)
            print(f'  Loaded {get_lhr_count_from(d1, d2)} flights')
        return self._flights_cache

    def _update_drift(self, symbol, model_fv, market_mid):
        from collections import deque
        if symbol not in self._drift_history:
            self._drift_history[symbol] = deque(maxlen=10)
            self._market_weight[symbol] = 0.2
        self._drift_history[symbol].append(model_fv - market_mid)
        hist = list(self._drift_history[symbol])
        if len(hist) >= 5:
            signs = [1 if x > 0 else -1 for x in hist[-5:]]
            if abs(sum(signs)) == 5:
                self._market_weight[symbol] = min(0.7, self._market_weight[symbol] + 0.1)
            else:
                self._market_weight[symbol] = max(0.1, self._market_weight[symbol] - 0.05)

    def bayesian_update(self, symbol, model_fv, model_std, market_mid):
        if market_mid is None:
            return model_fv, model_std
        self._update_drift(symbol, model_fv, market_mid)
        w = self._market_weight.get(symbol, 0.2)
        market_std = model_std * (2.0 - w)
        w_model  = 1.0 / model_std**2
        w_market = 1.0 / market_std**2
        post_fv  = (w_model * model_fv + w_market * market_mid) / (w_model + w_market)
        post_std = (1.0 / (w_model + w_market)) ** 0.5
        return post_fv, post_std

    def _hours_to_settle(self):
        settle  = LONDON_TZ.localize(self.SETTLE_TIME)
        now_lon = datetime.now(timezone.utc).astimezone(pytz.timezone('Europe/London'))
        return max((settle - now_lon).total_seconds() / 3600, 0.0)

    def wx_spot_fv(self):
        """
        改进3: std 用 forecast horizon 时间加权.
        越接近结算预报越准, std 随剩余时间收窄: std *= sqrt(h_left / total_h)
        """
        df = self._get_weather()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        start  = LONDON_TZ.localize(self.SESSION_START)
        df = df[(df['time'] >= start) & (df['time'] <= settle)].copy()
        df['temp_f'] = df['temperature'] * 9/5 + 32
        df['prod']   = df['temp_f'] * df['humidity']
        row = df[df['time'] == settle]
        if row.empty: row = df.iloc[[-1]]
        fv       = float(row['prod'].values[0])
        base_std = float(df['prod'].std()) if len(df) > 1 else fv * 0.02
        # 时间加权: 剩余时间越少, 预报越准, std 越小
        h_left = self._hours_to_settle()
        time_factor = max((h_left / self.TOTAL_HOURS) ** 0.5, 0.05)
        std = base_std * time_factor
        return round(fv), max(std, 1.0)

    def wx_sum_fv(self):
        df = self._get_weather()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        start  = LONDON_TZ.localize(self.SESSION_START)
        df = df[(df['time'] >= start) & (df['time'] <= settle)].copy()
        df['temp_f'] = df['temperature'] * 9/5 + 32
        df['prod']   = df['temp_f'] * df['humidity']
        now_lon = datetime.now(timezone.utc).astimezone(pytz.timezone('Europe/London'))
        observed    = float(df[df['time'] <= now_lon]['prod'].sum() / 100)
        forecast_df = df[df['time'] > now_lon]
        forecast    = float(forecast_df['prod'].sum() / 100)
        fv  = observed + forecast
        per_step_std = float(df['prod'].std()) / 100 if len(df) > 1 else 5.0
        std = per_step_std * max(len(forecast_df) ** 0.5, 1)
        return round(fv), max(std, 1.0)

    def tide_spot_fv(self):
        df, (coeffs, t0, res_std) = self._get_thames()
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        fv  = abs(predict_tidal_level(coeffs, t0, settle)) * 1000
        std = res_std * 1000
        return round(fv), max(std, 1.0)

    def tide_swing_fv(self):
        df, (coeffs, t0, res_std) = self._get_thames()
        start  = LONDON_TZ.localize(self.SESSION_START)
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        dr = pd.date_range(start=start, end=settle, freq='15min', tz='Europe/London')
        levels = predict_tidal_series(coeffs, t0, dr)
        df_sess = df[df['time'] >= start].copy()
        level_series = pd.Series(levels, index=dr)
        for _, row in df_sess.iterrows():
            if row['time'] in level_series.index:
                level_series[row['time']] = row['level']
        diff_cm = level_series.diff().abs() * 100
        payoff  = (np.maximum(0, 20 - diff_cm) + np.maximum(0, diff_cm - 25)).sum()
        fv = float(payoff)
        n_steps = len(dr)
        noise = np.random.normal(0, res_std, (1000, n_steps))
        noisy_levels  = levels[np.newaxis, :] + noise
        noisy_diff    = np.abs(np.diff(noisy_levels, axis=1)) * 100
        noisy_payoff  = (np.maximum(0, 20 - noisy_diff) + np.maximum(0, noisy_diff - 25)).sum(axis=1)
        std = float(noisy_payoff.std())
        return round(fv), max(std, 1.0)

    def lhr_count_fv(self):
        if self._flights_cache is None: return 1400, 50.0
        d1, d2 = self._flights_cache
        return round(float(get_lhr_count_from(d1, d2))), 30.0

    def lhr_index_fv(self):
        if self._flights_cache is None: return 50, 20.0
        d1, d2 = self._flights_cache
        return round(float(get_lhr_index_from(d1, d2))), 15.0

    def lon_etf_fv(self, ts=None, wx=None, lc=None):
        if ts is None: ts = self.tide_spot_fv()
        if wx is None: wx = self.wx_spot_fv()
        if lc is None: lc = self.lhr_count_fv()
        fv  = ts[0] + wx[0] + lc[0]
        std = (ts[1]**2 + wx[1]**2 + lc[1]**2) ** 0.5
        return round(fv), max(std, 1.0)

    def lon_fly_fv(self, ts=None, wx=None, lc=None):
        """分量蒙特卡洛: TIDE_SPOT~FoldedNormal, WX_SPOT~Normal, LHR_COUNT~Poisson"""
        if ts is None: ts = self.tide_spot_fv()
        if wx is None: wx = self.wx_spot_fv()
        if lc is None: lc = self.lhr_count_fv()
        N = self.MC_SAMPLES
        tide_samples = np.abs(np.random.normal(ts[0]/1000.0, ts[1]/1000.0, N)) * 1000
        wx_samples   = np.random.normal(wx[0], wx[1], N)
        lhr_samples  = np.random.poisson(max(lc[0], 1), N).astype(float)
        etf_samples  = np.maximum(0, tide_samples + wx_samples + lhr_samples)
        fly_samples  = (2*np.maximum(0, 6200 - etf_samples)
                        + np.maximum(0, etf_samples - 6200)
                        - 2*np.maximum(0, etf_samples - 6600)
                        + 3*np.maximum(0, etf_samples - 7000))
        return round(float(fly_samples.mean())), max(float(fly_samples.std()), 1.0)

    def lon_fly_greeks(self, ts=None, wx=None, lc=None):
        """
        改进6: 用 MC 数值差分估算 LON_FLY 的 delta 和 gamma w.r.t. LON_ETF.
        返回 (delta, gamma).
        """
        if ts is None: ts = self.tide_spot_fv()
        if wx is None: wx = self.wx_spot_fv()
        if lc is None: lc = self.lhr_count_fv()
        etf_fv  = ts[0] + wx[0] + lc[0]
        etf_std = (ts[1]**2 + wx[1]**2 + lc[1]**2) ** 0.5
        h = max(etf_std * 0.1, 10.0)   # bump size
        def fly_mc(etf_center):
            s = np.random.normal(etf_center, etf_std, self.MC_SAMPLES)
            s = np.maximum(0, s)
            return float((2*np.maximum(0,6200-s)+np.maximum(0,s-6200)
                          -2*np.maximum(0,s-6600)+3*np.maximum(0,s-7000)).mean())
        np.random.seed(42)
        f_up   = fly_mc(etf_fv + h)
        np.random.seed(42)
        f_mid  = fly_mc(etf_fv)
        np.random.seed(42)
        f_down = fly_mc(etf_fv - h)
        delta = (f_up - f_down) / (2 * h)
        gamma = (f_up - 2*f_mid + f_down) / (h**2)
        return delta, gamma

    def get_all(self, market_mids=None):
        if market_mids is None: market_mids = {}
        ts  = self.tide_spot_fv()
        tsw = self.tide_swing_fv()
        wx  = self.wx_spot_fv()
        wxs = self.wx_sum_fv()
        lc  = self.lhr_count_fv()
        li  = self.lhr_index_fv()
        etf = self.lon_etf_fv(ts=ts, wx=wx, lc=lc)
        fly = self.lon_fly_fv(ts=ts, wx=wx, lc=lc)
        raw = {'TIDE_SPOT':ts,'TIDE_SWING':tsw,'WX_SPOT':wx,'WX_SUM':wxs,
               'LHR_COUNT':lc,'LHR_INDEX':li,'LON_ETF':etf,'LON_FLY':fly}
        result = {}
        for sym, (fv, std) in raw.items():
            mid = market_mids.get(sym)
            post_fv, post_std = self.bayesian_update(sym, fv, std, mid)
            result[sym] = (post_fv, post_std)
        return result


In [ ]:
def get_lhr_count_from(d1, d2):
    return (len(d1.get('arrivals', [])) + len(d1.get('departures', []))
            + len(d2.get('arrivals', [])) + len(d2.get('departures', [])))

def get_lhr_index_from(d1, d2):
    date_range = pd.date_range(
        start=LONDON_TZ.localize(datetime(2026, 2, 28, 12, 0, 0)),
        end=LONDON_TZ.localize(datetime(2026, 3, 1, 12, 0, 0)),
        freq='30min', tz='Europe/London')
    idx = 0.0
    for i in range(24):
        arr, dep = group_flights(d1, date_range[i], date_range[i+1])
        idx += ((arr - dep) / (arr + dep)) * 100 if (arr + dep) else 0
    for i in range(24):
        arr, dep = group_flights(d2, date_range[i+24], date_range[i+25])
        idx += ((arr - dep) / (arr + dep)) * 100 if (arr + dep) else 0
    return round(abs(idx))


In [ ]:
from collections import deque

class AlgoBot(BaseBot):
    """
    五模块策略 v4:
      A. A-S 做市: gamma/k 动态校准 + reservation price
      B. ETF 套利: 腿间深度检查 + basis 止损
      C. 方向性: 连续衰减 + 成交量/spread 过滤
      D. LON_FLY delta+gamma 对冲 (折点附近加频)
      E. 组合 delta 汇总 + 相关性风险
    """
    SYMBOLS        = ['TIDE_SPOT','TIDE_SWING','WX_SPOT','WX_SUM',
                      'LHR_COUNT','LHR_INDEX','LON_ETF','LON_FLY']
    ETF_LEGS       = ['TIDE_SPOT', 'WX_SPOT', 'LHR_COUNT']
    POSITION_LIMIT = 100
    SETTLE_TIME    = datetime(2026, 3, 1, 12, 0, 0)

    MM_MIN_SPREAD    = 3
    MM_BASE_QTY      = 10
    MM_GAMMA_BASE    = 0.1    # A-S 风险厌恶基准, 按波动率动态缩放
    MM_K_BASE        = 1.5    # 市场深度基准, 按 orderbook 厚度动态调整
    REQUOTE_THRESH   = 2
    ARB_THRESH       = 25
    ARB_QTY          = 20
    ARB_BASIS_WINDOW = 50
    ARB_BASIS_STOP   = 60
    DIR_Z_THRESH     = 2.0
    DIR_QTY          = 15
    DIR_DECAY_N      = 3
    DIR_MIN_VOL_RATIO = 1.2   # 成交量需高于近期均值的倍数
    DIR_MAX_SPREAD_RATIO = 2.0  # spread 不能超过正常水平的倍数
    STOP_LOSS        = -5000
    UNWIND_MINS      = 30
    LOOP_INTERVAL    = 10
    CORR_PAIRS       = [('TIDE_SPOT','LON_ETF'), ('WX_SPOT','LON_ETF'),
                        ('LHR_COUNT','LON_ETF'), ('LON_ETF','LON_FLY')]
    CORR_NET_LIMIT   = 150
    # LON_FLY gamma 对冲: ETF 距折点多近时触发加频对冲
    FLY_KINK_POINTS  = [6200, 6600, 7000]
    FLY_KINK_BAND    = 50     # 距折点50以内视为高gamma区

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fve              = FairValueEngine()
        self.fair_values      = {}
        self.market_mids      = {}
        self.last_quotes      = {}
        self.active_orders    = {}
        self._arb_cooldown    = 0.0
        self._dir_cooldown    = {}
        self._mid_history     = {s: deque(maxlen=20) for s in self.SYMBOLS}
        self._vol_history     = {s: deque(maxlen=20) for s in self.SYMBOLS}  # 成交量历史
        self._dir_streak      = {s: 0 for s in self.SYMBOLS}
        self._basis_history   = deque(maxlen=self.ARB_BASIS_WINDOW)
        self._arb_position    = 0
        self._arb_entry_basis = None
        self._fly_delta       = 0.0
        self._fly_gamma       = 0.0
        self._last_hedge_ts   = 0.0

    # ── SSE 回调 ──────────────────────────────────────────────────────────
    def on_trades(self, trade):
        side = 'BOUGHT' if trade.buyer == self.username else 'SOLD'
        self._vol_history[trade.product].append(trade.volume)
        print(f'  FILL {side} {trade.volume}x {trade.product} @ {trade.price}')

    def on_orderbook(self, ob):
        mid = self._calc_mid(ob)
        if mid is not None:
            self.market_mids[ob.product] = mid
            self._mid_history[ob.product].append(mid)
        if ob.product in self.ETF_LEGS + ['LON_ETF']:
            self._realtime_arb_check()
        if ob.product in self.fair_values:
            self._realtime_dir_check(ob.product)

    # ── 工具方法 ──────────────────────────────────────────────────────────
    def _calc_mid(self, ob):
        bids = [o.price for o in ob.buy_orders  if o.volume - o.own_volume > 0]
        asks = [o.price for o in ob.sell_orders if o.volume - o.own_volume > 0]
        return (max(bids) + min(asks)) / 2 if bids and asks else None

    def _calc_spread(self, ob):
        bids = [o.price for o in ob.buy_orders  if o.volume - o.own_volume > 0]
        asks = [o.price for o in ob.sell_orders if o.volume - o.own_volume > 0]
        return (min(asks) - max(bids)) if bids and asks else None

    def _ob_depth(self, ob, qty):
        """检查 orderbook 买/卖方向是否有足够深度 >= qty, 返回 (bid_ok, ask_ok)"""
        bid_vol = sum(o.volume - o.own_volume for o in ob.buy_orders)
        ask_vol = sum(o.volume - o.own_volume for o in ob.sell_orders)
        return bid_vol >= qty, ask_vol >= qty

    def _send_ioc(self, order):
        resp = self.send_order(order)
        if resp and resp.status == 'ACTIVE':
            self.cancel_order(resp.id)
        return resp

    def _cancel_symbol(self, symbol):
        for oid in self.active_orders.get(symbol, []):
            self.cancel_order(oid)
        self.active_orders[symbol] = []

    def _mins_to_settle(self):
        settle = LONDON_TZ.localize(self.SETTLE_TIME)
        now    = datetime.now(pytz.timezone('Europe/London'))
        return (settle - now).total_seconds() / 60

    def _get_positions(self):
        return self.get_positions()

    def _short_vol(self, symbol):
        hist = list(self._mid_history[symbol])
        if len(hist) < 3:
            return self.fair_values.get(symbol, (0, self.MM_MIN_SPREAD * 2))[1]
        return max(float(np.std(hist)), self.MM_MIN_SPREAD)

    def _recent_trade_vol(self, symbol):
        hist = list(self._vol_history[symbol])
        return float(np.mean(hist)) if hist else 0.0

    # ── 模块 A: A-S 做市 (动态 gamma/k + reservation price) ──────────────
    def module_a_market_make(self, positions):
        """
        改进1:
        - gamma 按波动率动态缩放: 高波动率产品用更大 gamma (更宽 spread)
        - k 从 orderbook 厚度估算: 挂单越薄 k 越小 (spread 越宽)
        - bid/ask 围绕 reservation price r = fv - q*gamma*sigma^2*T 展开
        """
        mins_left = self._mins_to_settle()
        T = max(mins_left / (24 * 60), 1e-4)
        for symbol in self.SYMBOLS:
            if symbol not in self.fair_values:
                continue
            fv, model_std = self.fair_values[symbol]
            sigma = self._short_vol(symbol)
            pos   = positions.get(symbol, 0)
            q     = pos / self.POSITION_LIMIT

            # 动态 gamma: 波动率越高风险厌恶越强
            ref_vol = self.fair_values.get(symbol, (1, 1))[1]
            gamma = self.MM_GAMMA_BASE * max(sigma / max(ref_vol, 1.0), 0.5)
            gamma = min(gamma, 1.0)

            # 动态 k: 从 orderbook 厚度估算
            try:
                ob = self.get_orderbook(symbol)
                bid_depth = sum(o.volume - o.own_volume for o in ob.buy_orders)
                ask_depth = sum(o.volume - o.own_volume for o in ob.sell_orders)
                avg_depth = max((bid_depth + ask_depth) / 2, 1)
                k = self.MM_K_BASE * (avg_depth / 20.0)  # 归一化到标准深度20
                k = max(k, 0.5)
            except Exception:
                k = self.MM_K_BASE

            # Reservation price (A-S 核心)
            r = fv - q * gamma * sigma**2 * T

            # 最优 spread
            as_spread   = (gamma * sigma**2 * T
                           + (2.0 / gamma) * np.log(1 + gamma / k))
            half_spread = max(self.MM_MIN_SPREAD, as_spread / 2)

            bid_price = round(r - half_spread)
            ask_price = round(r + half_spread)
            if bid_price <= 0 or ask_price <= bid_price:
                continue
            last = self.last_quotes.get(symbol)
            if (last and abs(last[0] - bid_price) < self.REQUOTE_THRESH
                    and abs(last[1] - ask_price) < self.REQUOTE_THRESH):
                continue
            self._cancel_symbol(symbol)
            avail_buy  = max(0, self.POSITION_LIMIT - pos)
            avail_sell = max(0, self.POSITION_LIMIT + pos)
            new_ids = []
            if avail_buy >= 1:
                qty = min(self.MM_BASE_QTY, avail_buy)
                r2 = self.send_order(OrderRequest(symbol, bid_price, Side.BUY, qty))
                if r2: new_ids.append(r2.id)
            if avail_sell >= 1:
                qty = min(self.MM_BASE_QTY, avail_sell)
                r2 = self.send_order(OrderRequest(symbol, ask_price, Side.SELL, qty))
                if r2: new_ids.append(r2.id)
            self.active_orders[symbol] = new_ids
            self.last_quotes[symbol]   = (bid_price, ask_price)
    # ── 模块 B: ETF 套利 (腿间深度检查) ──────────────────────────────────
    def _realtime_arb_check(self):
        if time.time() - self._arb_cooldown < 2.0:
            return
        etf_mid  = self.market_mids.get('LON_ETF')
        leg_mids = [self.market_mids.get(s) for s in self.ETF_LEGS]
        if etf_mid is None or any(m is None for m in leg_mids):
            return
        basis = etf_mid - sum(leg_mids)
        self._basis_history.append(basis)
        if len(self._basis_history) >= 10:
            b_arr      = np.array(self._basis_history)
            dyn_thresh = max(self.ARB_THRESH, float(np.mean(b_arr) + 2*np.std(b_arr)))
        else:
            dyn_thresh = self.ARB_THRESH
        if abs(basis) > dyn_thresh:
            positions = self._get_positions()
            self._execute_arb(basis, positions)
            self._arb_cooldown = time.time()

    def _arb_basis_stop_check(self, positions):
        if self._arb_position == 0 or self._arb_entry_basis is None:
            return
        etf_mid  = self.market_mids.get('LON_ETF')
        leg_mids = [self.market_mids.get(s) for s in self.ETF_LEGS]
        if etf_mid is None or any(m is None for m in leg_mids):
            return
        current_basis = etf_mid - sum(leg_mids)
        adverse = (
            (self._arb_position > 0 and current_basis > self._arb_entry_basis + self.ARB_BASIS_STOP) or
            (self._arb_position < 0 and current_basis < self._arb_entry_basis - self.ARB_BASIS_STOP)
        )
        if adverse:
            print(f'  ARB BASIS STOP: entry={self._arb_entry_basis:.1f} current={current_basis:.1f}')
            self._close_arb_position(positions)
            self._arb_position    = 0
            self._arb_entry_basis = None

    def _close_arb_position(self, positions):
        etf_pos = positions.get('LON_ETF', 0)
        if etf_pos != 0:
            ob = self.get_orderbook('LON_ETF')
            if etf_pos > 0 and ob.buy_orders:
                self._send_ioc(OrderRequest('LON_ETF', max(o.price for o in ob.buy_orders), Side.SELL, abs(etf_pos)))
            elif etf_pos < 0 and ob.sell_orders:
                self._send_ioc(OrderRequest('LON_ETF', min(o.price for o in ob.sell_orders), Side.BUY, abs(etf_pos)))
        for s in self.ETF_LEGS:
            leg_pos = positions.get(s, 0)
            if leg_pos == 0: continue
            ob = self.get_orderbook(s)
            if leg_pos > 0 and ob.buy_orders:
                self._send_ioc(OrderRequest(s, max(o.price for o in ob.buy_orders), Side.SELL, abs(leg_pos)))
            elif leg_pos < 0 and ob.sell_orders:
                self._send_ioc(OrderRequest(s, min(o.price for o in ob.sell_orders), Side.BUY, abs(leg_pos)))

    def _execute_arb(self, basis, positions):
        """
        改进2: 先检查所有4条腿的 orderbook 深度是否满足 qty,
        全部满足才发单, 避免腿间风险.
        """
        if basis > self.ARB_THRESH:
            qty = min(self.ARB_QTY,
                      self.POSITION_LIMIT + positions.get('LON_ETF', 0),
                      min(self.POSITION_LIMIT - positions.get(s, 0) for s in self.ETF_LEGS))
            if qty < 1: return
            # 深度检查: ETF 需要 bid 深度, 各腿需要 ask 深度
            ob_etf = self.get_orderbook('LON_ETF')
            bid_ok, _ = self._ob_depth(ob_etf, qty)
            if not bid_ok:
                print(f'  ARB SKIP: ETF bid depth insufficient')
                return
            leg_obs = {s: self.get_orderbook(s) for s in self.ETF_LEGS}
            for s, ob in leg_obs.items():
                _, ask_ok = self._ob_depth(ob, qty)
                if not ask_ok:
                    print(f'  ARB SKIP: {s} ask depth insufficient')
                    return
            print(f'  ARB SELL ETF basis={basis:.1f} qty={qty}')
            if ob_etf.buy_orders:
                self._send_ioc(OrderRequest('LON_ETF', max(o.price for o in ob_etf.buy_orders), Side.SELL, qty))
            for s, ob in leg_obs.items():
                if ob.sell_orders:
                    self._send_ioc(OrderRequest(s, min(o.price for o in ob.sell_orders), Side.BUY, qty))
            self._arb_position    = 1
            self._arb_entry_basis = basis
        else:
            qty = min(self.ARB_QTY,
                      self.POSITION_LIMIT - positions.get('LON_ETF', 0),
                      min(self.POSITION_LIMIT + positions.get(s, 0) for s in self.ETF_LEGS))
            if qty < 1: return
            # 深度检查: ETF 需要 ask 深度, 各腿需要 bid 深度
            ob_etf = self.get_orderbook('LON_ETF')
            _, ask_ok = self._ob_depth(ob_etf, qty)
            if not ask_ok:
                print(f'  ARB SKIP: ETF ask depth insufficient')
                return
            leg_obs = {s: self.get_orderbook(s) for s in self.ETF_LEGS}
            for s, ob in leg_obs.items():
                bid_ok, _ = self._ob_depth(ob, qty)
                if not bid_ok:
                    print(f'  ARB SKIP: {s} bid depth insufficient')
                    return
            print(f'  ARB BUY ETF basis={basis:.1f} qty={qty}')
            if ob_etf.sell_orders:
                self._send_ioc(OrderRequest('LON_ETF', min(o.price for o in ob_etf.sell_orders), Side.BUY, qty))
            for s, ob in leg_obs.items():
                if ob.buy_orders:
                    self._send_ioc(OrderRequest(s, max(o.price for o in ob.buy_orders), Side.SELL, qty))
            self._arb_position    = -1
            self._arb_entry_basis = basis

    def module_b_arbitrage(self, positions):
        self._arb_basis_stop_check(positions)
        etf_mid  = self.market_mids.get('LON_ETF')
        leg_mids = [self.market_mids.get(s) for s in self.ETF_LEGS]
        if etf_mid is None or any(m is None for m in leg_mids):
            return
        basis = etf_mid - sum(leg_mids)
        self._basis_history.append(basis)
        if abs(basis) > self.ARB_THRESH:
            self._execute_arb(basis, positions)

    # ── 模块 C: 方向性 (成交量+spread 过滤) ──────────────────────────────
    def _realtime_dir_check(self, symbol):
        if time.time() - self._dir_cooldown.get(symbol, 0) < 5.0:
            return
        if symbol not in self.fair_values:
            return
        fv, std = self.fair_values[symbol]
        mid = self.market_mids.get(symbol)
        if mid is None or std <= 0:
            return
        z = (mid - fv) / std
        if abs(z) > self.DIR_Z_THRESH:
            positions = self._get_positions()
            self._execute_directional(symbol, z, fv, mid, std, positions)
            self._dir_cooldown[symbol] = time.time()

    def _dir_signal_valid(self, symbol):
        """
        改进4: 方向性信号质量过滤.
        - 成交量需高于近期均值 DIR_MIN_VOL_RATIO 倍
        - 当前 spread 不能超过正常水平 DIR_MAX_SPREAD_RATIO 倍
        """
        try:
            ob = self.get_orderbook(symbol)
            cur_spread = self._calc_spread(ob)
            # 正常 spread 估算: 用 fair value std 的一半
            fv, std = self.fair_values.get(symbol, (0, self.MM_MIN_SPREAD * 2))
            normal_spread = max(std * 0.5, self.MM_MIN_SPREAD)
            if cur_spread is not None and cur_spread > normal_spread * self.DIR_MAX_SPREAD_RATIO:
                return False  # spread 异常宽, 市场不确定性高
        except Exception:
            pass
        # 成交量过滤: 近期有成交才认为信号有效
        avg_vol = self._recent_trade_vol(symbol)
        if avg_vol < 1.0:
            return True  # 没有历史成交量数据时不过滤
        recent = list(self._vol_history[symbol])
        if not recent:
            return True
        latest_vol = recent[-1] if recent else 0
        return latest_vol >= avg_vol * self.DIR_MIN_VOL_RATIO

    def _execute_directional(self, symbol, z, fv, mid, std, positions):
        if not self._dir_signal_valid(symbol):
            return
        pos    = positions.get(symbol, 0)
        streak = self._dir_streak.get(symbol, 0)
        base_qty = self.DIR_QTY
        if z > 0 and streak >= self.DIR_DECAY_N:
            base_qty = max(1, base_qty // (2 ** (streak // self.DIR_DECAY_N)))
        elif z < 0 and streak <= -self.DIR_DECAY_N:
            base_qty = max(1, base_qty // (2 ** (abs(streak) // self.DIR_DECAY_N)))
        if z > self.DIR_Z_THRESH:
            qty = min(base_qty, self.POSITION_LIMIT + pos)
            if qty < 1:
                self._dir_streak[symbol] = 0
                return
            ob = self.get_orderbook(symbol)
            if ob.buy_orders:
                print(f'  DIR SELL {symbol} z={z:.2f} qty={qty} streak={streak}')
                self._send_ioc(OrderRequest(symbol, max(o.price for o in ob.buy_orders), Side.SELL, qty))
            self._dir_streak[symbol] = max(0, streak) + 1
        elif z < -self.DIR_Z_THRESH:
            qty = min(base_qty, self.POSITION_LIMIT - pos)
            if qty < 1:
                self._dir_streak[symbol] = 0
                return
            ob = self.get_orderbook(symbol)
            if ob.sell_orders:
                print(f'  DIR BUY  {symbol} z={z:.2f} qty={qty} streak={streak}')
                self._send_ioc(OrderRequest(symbol, min(o.price for o in ob.sell_orders), Side.BUY, qty))
            self._dir_streak[symbol] = min(0, streak) - 1
        else:
            self._dir_streak[symbol] = 0

    def module_c_directional(self, positions):
        for symbol in self.SYMBOLS:
            if symbol not in self.fair_values: continue
            fv, std = self.fair_values[symbol]
            mid = self.market_mids.get(symbol)
            if mid is None or std <= 0: continue
            z = (mid - fv) / std
            if abs(z) > self.DIR_Z_THRESH:
                self._execute_directional(symbol, z, fv, mid, std, positions)

    # ── 模块 D: LON_FLY delta+gamma 对冲 ─────────────────────────────────
    def module_d_fly_hedge(self, positions):
        """
        改进6: 追踪 delta 和 gamma.
        - 折点附近 (高 gamma 区) 降低对冲触发阈值, 更频繁对冲
        - 正常区域维持原有阈值
        """
        fly_pos = positions.get('LON_FLY', 0)
        if fly_pos == 0: return
        etf_mid = self.market_mids.get('LON_ETF')
        if etf_mid is None: return

        # 判断是否在高 gamma 区 (折点附近)
        near_kink = any(abs(etf_mid - k) < self.FLY_KINK_BAND for k in self.FLY_KINK_POINTS)
        hedge_threshold = 2 if near_kink else 5   # 折点附近更频繁对冲
        hedge_cooldown  = 5.0 if near_kink else 30.0

        if time.time() - self._last_hedge_ts < hedge_cooldown:
            return

        # 用 FairValueEngine 计算 delta
        try:
            delta, gamma = self.fve.lon_fly_greeks()
            self._fly_delta = delta
            self._fly_gamma = gamma
        except Exception:
            # fallback: 分段线性 delta
            if   etf_mid < 6200: delta = -2.0
            elif etf_mid < 6600: delta = -1.0
            elif etf_mid < 7000: delta =  1.0
            else:                delta = -2.0
            self._fly_delta = delta

        target_hedge = -fly_pos * self._fly_delta
        current_etf  = positions.get('LON_ETF', 0)
        delta_needed = target_hedge - current_etf

        if abs(delta_needed) < hedge_threshold: return
        qty = min(abs(int(delta_needed)), self.POSITION_LIMIT - abs(current_etf))
        if qty < 1: return
        ob = self.get_orderbook('LON_ETF')
        if delta_needed > 0 and ob.sell_orders:
            print(f'  HEDGE BUY LON_ETF {qty} (near_kink={near_kink} delta={self._fly_delta:.2f})')
            self._send_ioc(OrderRequest('LON_ETF', min(o.price for o in ob.sell_orders), Side.BUY, qty))
        elif delta_needed < 0 and ob.buy_orders:
            print(f'  HEDGE SELL LON_ETF {qty} (near_kink={near_kink} delta={self._fly_delta:.2f})')
            self._send_ioc(OrderRequest('LON_ETF', max(o.price for o in ob.buy_orders), Side.SELL, qty))
        self._last_hedge_ts = time.time()

    # ── 模块 E: 组合 delta 汇总 + 相关性风险 ─────────────────────────────
    def module_e_portfolio_risk(self, positions):
        total_notional = sum(
            abs(pos) * (self.fair_values.get(sym, (0, 0))[0] or 0)
            for sym, pos in positions.items()
        )
        print(f'  PORTFOLIO delta={positions} notional~{total_notional:.0f} fly_delta={self._fly_delta:.2f} fly_gamma={self._fly_gamma:.4f}')
        for sym_a, sym_b in self.CORR_PAIRS:
            pos_a = positions.get(sym_a, 0)
            pos_b = positions.get(sym_b, 0)
            net   = abs(pos_a) + abs(pos_b)
            if net > self.CORR_NET_LIMIT:
                print(f'  CORR RISK {sym_a}/{sym_b} net={net} > {self.CORR_NET_LIMIT}, reducing')
                target     = sym_a if abs(pos_a) >= abs(pos_b) else sym_b
                tpos       = positions.get(target, 0)
                reduce_qty = min(abs(tpos), net - self.CORR_NET_LIMIT)
                if reduce_qty < 1: continue
                ob = self.get_orderbook(target)
                if tpos > 0 and ob.buy_orders:
                    self._send_ioc(OrderRequest(target, max(o.price for o in ob.buy_orders), Side.SELL, reduce_qty))
                elif tpos < 0 and ob.sell_orders:
                    self._send_ioc(OrderRequest(target, min(o.price for o in ob.sell_orders), Side.BUY, reduce_qty))

    # ── 结算前平仓 ────────────────────────────────────────────────────────
    def unwind_all(self, positions):
        print('  UNWIND: closing all positions before settlement')
        self.cancel_all_orders()
        for symbol, pos in positions.items():
            if pos == 0: continue
            ob = self.get_orderbook(symbol)
            if pos > 0 and ob.buy_orders:
                self._send_ioc(OrderRequest(symbol, max(o.price for o in ob.buy_orders), Side.SELL, abs(pos)))
            elif pos < 0 and ob.sell_orders:
                self._send_ioc(OrderRequest(symbol, min(o.price for o in ob.sell_orders), Side.BUY, abs(pos)))

    # ── 主循环 ────────────────────────────────────────────────────────────
    def run_bot(self):
        print('Initialising...')
        self.fve.init_flights()
        self.start()
        print('Bot running. Ctrl+C to stop.')
        try:
            while True:
                t0 = time.time()
                mins_left = self._mins_to_settle()

                if mins_left <= self.UNWIND_MINS:
                    positions = self._get_positions()
                    self.unwind_all(positions)
                    print(f'  {mins_left:.1f} mins to settle, unwinding. Sleeping 60s.')
                    time.sleep(60)
                    continue

                for sym in self.SYMBOLS:
                    try:
                        ob  = self.get_orderbook(sym)
                        mid = self._calc_mid(ob)
                        if mid:
                            self.market_mids[sym] = mid
                            self._mid_history[sym].append(mid)
                    except Exception:
                        pass

                self.fair_values = self.fve.get_all(market_mids=self.market_mids)
                fv_str = {s: f'{v[0]}+-{v[1]:.0f}' for s, v in self.fair_values.items()}
                print(f'\n[{datetime.now().strftime("%H:%M:%S")}] {mins_left:.0f}min left')
                print(f'  FV: {fv_str}')

                positions = self._get_positions()
                pnl = self.get_pnl()
                print(f'  Pos: {positions}  PnL: {pnl}')

                if pnl.get('unrealizedPnl', 0) < self.STOP_LOSS:
                    print(f'  STOP LOSS triggered at {pnl}')
                    self.cancel_all_orders()
                    self.stop()
                    break

                self.module_e_portfolio_risk(positions)
                self.module_b_arbitrage(positions)
                self.module_c_directional(positions)
                self.module_d_fly_hedge(positions)
                self.module_a_market_make(positions)

                elapsed = time.time() - t0
                time.sleep(max(1, self.LOOP_INTERVAL - elapsed))

        except KeyboardInterrupt:
            print('\nStopping...')
            self.cancel_all_orders()
            self.stop()
            print('Final positions:', self.get_positions())
            print('Final PnL:', self.get_pnl())
        except Exception as e:
            import traceback
            traceback.print_exc()
            self.cancel_all_orders()
            self.stop()


In [ ]:
bot = AlgoBot(EXCHANGE_URL, USERNAME, PASSWORD)
bot.run_bot()
